In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes "huggingface-hub<1.0,>=0.24.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.9 MB/s eta 0:00:00


In [ ]:
!pip install -q -U "huggingface-hub>=0.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 24.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.46.3 requires huggingface-hub<1.0,>=0.23.2, but you have huggingface-hub 1.31.0 which is incompatible.
tokenizers 0.20.3 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.31.0 which is incompatible.


In [ ]:
import torch
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import json
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"GPU Device: {torch.cuda.get_device_name(0)}")
!nvidia-smi

CUDA Available: True
GPU Device: Tesla T4
Sat Sep 12 18:51:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----

In [ ]:


MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)

for name, param in model.named_parameters():
  if "norm" in name or param.requires_grad:
    param.data = param.data.to(torch.float32)

model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
from datasets import load_dataset

DATA_DIR = "/customer-support-ai/data/processed"

dataset = load_dataset(
    "json",
    data_files={"train": f"{DATA_DIR}/train.jsonl", "val": f"{DATA_DIR}/val.jsonl"},
)


def format_prompts(batch):
  formatted_texts = []
  for messages in batch["messages"]:
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    formatted_texts.append(text)
  return {"text": formatted_texts}


dataset = dataset.map(format_prompts, batched=True)
print("Sample Formatted Prompt:\n", dataset["train"][0]["text"])

Sample Formatted Prompt:
 <|im_start|>system
شما موتور تحلیل پیام مشتریان customer-support-ai هستید. پیام کاربر را تحلیل کنید و خروجی را دقیقاً در قالب JSON مشخص‌شده برگردانید.<|im_end|>
<|im_start|>user
اگه کالا رو بعد از ۷ روز برگردونم چی میشه؟<|im_end|>
<|im_start|>assistant
{"intent": "refund_policy", "entities": {}, "requires_tool": true, "tool": "rag_policy_search", "requires_confirmation": false, "needs_clarification": false}<|im_end|>



In [ ]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = (
    "/customer-support-ai/models/qwen2.5-support-lora"
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=512,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=False,  
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print("مدل با موفقیت آموزش داده و ذخیره شد!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.348700,0.330551,0.334351,60616.000000,0.913675
100,0.307000,0.282842,0.294791,120389.000000,0.926632
150,0.281700,0.267749,0.262828,180636.000000,0.928735


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.348700,0.330551,0.334351,60616.000000,0.913675
100,0.307000,0.282842,0.294791,120389.000000,0.926632
150,0.281700,0.267749,0.262828,180636.000000,0.928735
200,0.238400,0.256286,0.245072,240209.000000,0.931286
250,0.205900,0.252635,0.221911,300343.000000,0.933204
300,0.218500,0.242404,0.227073,360398.000000,0.934471
350,0.218100,0.238399,0.223490,420631.000000,0.935183
400,0.180200,0.239628,0.199234,480640.000000,0.935697
450,0.176200,0.239973,0.196087,540911.000000,0.936080
500,0.163200,0.239028,0.193518,600885.000000,0.936712


مدل با موفقیت آموزش داده و ذخیره شد!


In [ ]:

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH = "/customer-support-ai/models/qwen2.5-support-lora/final_adapter"
MERGED_PATH = "/merged_model"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",  
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.merge_and_unload()

model.save_pretrained(MERGED_PATH)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(MERGED_PATH)


در حال لود مدل پایه...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

در حال اتصال آداپتر و ادغام...
در حال ذخیره مدل یکپارچه...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

مدل با موفقیت Merge شد!


In [ ]:
!pip install -q gguf

!python llama.cpp/convert_hf_to_gguf.py /merged_model \
  --outfile /customer-support-ai/models/qwen_support_q8.gguf \
  --outtype q8_0



INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> Q8_0, shape = {2048, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> Q8_0, shape = {11008, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> Q8_0, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> Q8_0, shape = {2048, 11008}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.fl